# Boston Neighbourhood House Price Prediction
### Housing Dataset | Regression

Predict the median value of owner-occupied homes in Boston neighbourhoods
using socioeconomic, environmental, and structural features.

**Dataset:** Housing.csv — 506 samples | 13 features
**Target:** MEDV (Median Home Value in $1,000s)
**Problem Type:** Regression
**Approach:** CRISP-DM Methodology

## 0. Business Understanding

### Background
Accurate property valuation is critical for multiple stakeholders in the real estate
market — buyers, sellers, mortgage lenders, and urban planners. Overvalued properties
lead to bad loans; undervalued ones cause financial loss for sellers and tax authorities.
The Boston Housing dataset captures socioeconomic and environmental factors that
influence home prices across different city neighbourhoods.

### Business Problem
A real estate analytics firm wants to build a model that can estimate the median
home value of any Boston neighbourhood given its characteristics. This model will:
- Support automated property valuation for mortgage applications
- Help buyers identify undervalued neighbourhoods
- Assist urban planners in understanding how factors like crime rate and school
  quality (PTRATIO) affect housing prices

### ML Objective
Build a **regression model** that predicts `MEDV` (median home value in $1,000s)
from 13 neighbourhood features including crime rate, room count, tax rate,
proximity to employment centres, and pollution levels.

### Success Criteria
- **R² Score** is the primary metric — a value above 0.80 on the test set
  indicates the model explains over 80% of price variance
- Train R² and Test R² should be within ~5% to confirm no overfitting
- Predicted price must be interpretable and reversible from log scale using np.expm1()

In [1]:
# Import core libraries for data manipulation, visualisation and numerical operations
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Data Loading
Loading the Boston Housing dataset and previewing its structure.

In [2]:
# Load the Boston Housing dataset and preview the first few rows
df=pd.read_csv("Housing.csv")
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1.0,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2.0,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2.0,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3.0,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3.0,222,18.7,396.90,5.33,36.2


## 2. Data Understanding
Exploring shape, column names, data types and unique values per column.

In [3]:
# Check dataset dimensions: (rows, columns)
df.shape

(509, 14)

In [4]:
# Inspect column names, data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 509 entries, 0 to 508
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     509 non-null    float64
 1   ZN       509 non-null    float64
 2   INDUS    506 non-null    float64
 3   CHAS     509 non-null    int64  
 4   NOX      507 non-null    float64
 5   RM       509 non-null    float64
 6   AGE      508 non-null    float64
 7   DIS      509 non-null    float64
 8   RAD      508 non-null    float64
 9   TAX      509 non-null    int64  
 10  PTRATIO  509 non-null    float64
 11  B        509 non-null    float64
 12  LSTAT    508 non-null    float64
 13  MEDV     509 non-null    float64
dtypes: float64(12), int64(2)
memory usage: 55.8 KB


### 2.1 Categorical Column Check
CHAS is a binary column (0=not on Charles River, 1=on Charles River).
RAD is an index of accessibility to radial highways (1-24).
Both are checked for unexpected values.

In [5]:
# Store column names as a list for reuse in downstream steps
col=df.columns.tolist()

In [6]:
# Count unique values per column to identify low-cardinality / categorical columns
df[col].nunique()

CRIM       503
ZN          26
INDUS       76
CHAS         2
NOX         81
RM         446
AGE        354
DIS        411
RAD          9
TAX         66
PTRATIO     46
B          356
LSTAT      453
MEDV       228
dtype: int64

In [7]:
# CHAS is binary (0=not on Charles River, 1=on Charles River) — check for unexpected values
df['CHAS'].value_counts()

CHAS
0    474
1     35
Name: count, dtype: int64

In [8]:
# RAD is an index of highway accessibility (1-24) — check for unexpected values
df['RAD'].value_counts()

RAD
24.0    134
5.0     115
4.0     109
3.0      38
6.0      27
8.0      24
2.0      24
1.0      20
7.0      17
Name: count, dtype: int64

## 3. Data Cleaning

### 3.1 Duplicate Values
Checking for repeated rows in the dataset.

In [9]:
# Count duplicate rows — any found will be dropped in the next cell
df.duplicated().sum()

np.int64(4)

In [10]:
# Remove duplicate rows from the dataset
df = df.drop_duplicates()

In [11]:
# Confirm zero duplicates remain after removal
df.duplicated().sum()

np.int64(0)

### 3.2 Null Values
Checking for missing values across all columns.

In [12]:
# Check for missing values across all 14 columns
df.isnull().sum()

CRIM       0
ZN         0
INDUS      3
CHAS       0
NOX        2
RM         0
AGE        1
DIS        0
RAD        1
TAX        0
PTRATIO    0
B          0
LSTAT      1
MEDV       0
dtype: int64

### 3.3 Fill Missing Values
Filling missing values with column median.
Median is preferred over mean as it is robust to outliers.

In [13]:
# Fill missing values with the column median
# Median is preferred over mean as it is robust to outliers
df.fillna(df.median(), inplace=True)

In [14]:
# Confirm all missing values have been filled
df.isnull().sum()

CRIM       0
ZN         0
INDUS      0
CHAS       0
NOX        0
RM         0
AGE        0
DIS        0
RAD        0
TAX        0
PTRATIO    0
B          0
LSTAT      0
MEDV       0
dtype: int64

## 4. Exploratory Data Analysis (EDA)

### 4.1 Column Classification
Grouping columns into continuous, discrete and categorical for targeted analysis.

| Group | Columns |
|---|---|
| Continuous | CRIM, ZN, INDUS, NOX, RM, AGE, DIS, PTRATIO, B, LSTAT, MEDV |
| Discrete | RAD, TAX |
| Categorical | CHAS |

In [16]:
# Group columns by type for targeted EDA and transformation
# Continuous: standard numeric | Discrete: integer-coded | Categorical: binary flag
continous = ['CRIM','ZN','INDUS','NOX','RM','AGE','DIS','PTRATIO','B','LSTAT','MEDV']
Discrete = ['RAD','TAX']
Categorical =  ['CHAS']

In [17]:
# Summary statistics for continuous features
df[continous].describe()

,CRIM,ZN,INDUS,NOX,RM,AGE,DIS,PTRATIO,B,LSTAT,MEDV
count,505.000000,505.000000,505.000000,505.000000,505.000000,505.000000,505.000000,505.000000,505.000000,505.000000,505.000000
mean,3.606091,11.386139,11.151307,0.554679,6.284816,68.534158,3.798725,18.452079,357.188772,12.654079,22.555644
std,8.608447,23.340080,6.843323,0.115693,0.703302,28.162574,2.106167,2.165696,90.647420,7.121604,9.191851
min,0.006320,0.000000,0.460000,0.385000,3.561000,2.900000,1.129600,12.600000,0.320000,1.730000,5.000000
25%,0.081990,0.000000,5.190000,0.449000,5.885000,45.000000,2.100700,17.400000,375.520000,7.010000,17.100000
50%,0.253870,0.000000,9.690000,0.538000,6.209000,77.150000,3.215700,19.000000,391.450000,11.360000,21.200000
75%,3.673670,12.500000,18.100000,0.624000,6.625000,94.100000,5.211900,20.200000,396.230000,16.940000,25.000000
max,88.976200,100.000000,27.740000,0.871000,8.780000,100.000000,12.126500,22.000000,396.900000,37.970000,50.000000


In [18]:
# Summary statistics for discrete features (RAD, TAX)
df[Discrete].describe()

,RAD,TAX
count,505.000000,505.000000
mean,9.522772,407.726733
std,8.690899,168.312294
min,1.000000,187.000000
25%,4.000000,279.000000
50%,5.000000,330.000000
75%,24.000000,666.000000
max,24.000000,711.000000


In [19]:
# Check skewness of continuous features — values above 0.5 or below -0.5 indicate skew
df[continous].skew()

CRIM       5.223410
ZN         2.222377
INDUS      0.294479
NOX        0.735879
RM         0.402463
AGE       -0.595617
DIS        1.009242
PTRATIO   -0.799325
B         -2.928761
LSTAT      0.918150
MEDV       1.108662
dtype: float64

### 4.2 Skewness Analysis
Checking skewness of continuous features.
Values above 0.5 or below -0.5 require transformation to normalize the distribution.

### 4.3 Log Transformation
Applying log1p transformation to highly skewed features.
log1p is used instead of log to safely handle zero values (log(0) is undefined).
MEDV (target) is also transformed — prediction will need np.expm1() to reverse it.

In [20]:
# Apply log1p to highly skewed features: CRIM, ZN, DIS
# B is right-skewed after reflection (B_max - B), so log1p applied to the flipped version
# MEDV (target) also log-transformed — use np.expm1() to reverse predictions
df['CRIM'] = np.log1p(df['CRIM'])
df['ZN'] = np.log1p(df['ZN'])
df['DIS'] = np.log1p(df['DIS'])

B_max = df['B'].max()
df['B'] = np.log1p(B_max - df['B'])

df['MEDV'] = np.log1p(df['MEDV'])

In [21]:
# Confirm skewness has reduced after log1p transformation
print(df[['CRIM','ZN','DIS','B','MEDV']].skew())

CRIM    1.277625
ZN      1.190406
DIS     0.328205
B       0.603021
MEDV   -0.244536
dtype: float64


### 4.4 Cube Root Transformation
CRIM and ZN still skewed after log transform — applying cube root (cbrt) as a stronger fix.
cbrt handles both positive and negative values unlike log.

In [22]:
# CRIM and ZN still skewed after log — apply cube root as a stronger correction
# cbrt handles both positive and negative values safely unlike log
df['CRIM'] = np.cbrt(df['CRIM'])
df['ZN'] = np.cbrt(df['ZN'])

print(df[['CRIM','ZN']].skew())

CRIM    0.551900
ZN      1.080547
dtype: float64


### 4.5 Outlier Detection (IQR Method)
Checking outliers across all continuous features using the IQR method.
Note: Outliers are not capped here — regression models like Ridge and Lasso
handle outliers through regularization rather than removing them.

In [24]:
# Detect remaining outliers using IQR method across all continuous columns
Q1 = df[continous].quantile(0.25)
Q3 = df[continous].quantile(0.75)

IQR = Q3 - Q1

outliers = ((df[continous] < (Q1 - 1.5*IQR)) |
            (df[continous] > (Q3 + 1.5*IQR)))

print(outliers.sum())

CRIM        0
ZN          0
INDUS       0
NOX         0
RM         30
AGE         0
DIS         0
PTRATIO    15
B           0
LSTAT       7
MEDV       47
dtype: int64


## 5. Feature Engineering

### 5.1 Define Features and Target
Separating features (X) and target variable (y = MEDV).
MEDV has already been log-transformed — predictions will be reversed using np.expm1().

In [25]:
# Separate features (X) from target (y)
X = df.drop('MEDV', axis=1)
y = df['MEDV']

### 5.2 Train Test Split
Splitting data into 80% training and 20% testing sets.
random_state=42 ensures the same split every time the notebook is run.

In [26]:
# Split into train (80%) and test (20%) sets with fixed random_state for reproducibility
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### 5.3 Feature Scaling
Applying StandardScaler to bring all features to mean=0 and std=1.
fit_transform on train data only — transform applied to test to prevent data leakage.

In [27]:
# Standardise features to mean=0 and std=1
# fit_transform on train only — transform on test to prevent data leakage
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 6. Modelling
Training and comparing 7 regression models.
Evaluation metric: R² Score (closer to 1.0 = better fit).

### 6.1 Linear Regression
Baseline regression model — assumes a linear relationship between features and MEDV.
Intercept and coefficients printed to show feature weights.

In [28]:
# Fit a baseline Linear Regression model and print the intercept and coefficients
from sklearn.linear_model import LinearRegression
model=LinearRegression()
model.fit(X_train,y_train)

print("Intercept:",model.intercept_)
print("Coefficients:",model.coef_)

Intercept: 3.0891510213245055
Coefficients: [-0.03997588  0.00629353  0.02214581  0.02963007 -0.08459328  0.07229645
  0.00031073 -0.1154594   0.06444092 -0.08712204 -0.07070241 -0.00579652
 -0.23680301]


In [29]:
#prediction on train data
ypred_train=model.predict(X_train)

#Evaluation on train data
from sklearn.metrics import r2_score
print("Train R2:",r2_score(y_train,ypred_train))

#cross Validation on train data
from sklearn.model_selection import cross_val_score
print("cross Validation score:",cross_val_score(model,X_train,y_train,cv=5).mean())

#prediction on test data
ypred_test = model.predict(X_test)

from sklearn.metrics import r2_score
print("test R2:",r2_score(y_test,ypred_test))

Train R2: 0.7715330371756268
cross Validation score: 0.7388329875491488
test R2: 0.6544009762343967


In [30]:
# OLS regression via statsmodels — provides detailed statistical summary (p-values, R², F-stat)
# Note: this uses the log-transformed, scaled training data for consistency with other models
import statsmodels.api as sm

# Add constant for intercept term
X_train_sm = sm.add_constant(X_train)

# Fit OLS model
ols_model = sm.OLS(y_train, X_train_sm).fit()

print(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                y_train   R-squared:                       0.772
Model:                            OLS   Adj. R-squared:                  0.764
Method:                 Least Squares   F-statistic:                     101.3
Date:                Sun, 15 Mar 2026   Prob (F-statistic):          3.39e-116
Time:                        02:36:55   Log-Likelihood:                 94.897
No. Observations:                 404   AIC:                            -161.8
Df Residuals:                     390   BIC:                            -105.8
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       3.0892      0.010    318.876      0.000       3.070       3.108
X_train[0]     -0.0400      0.030     -1.337      0.182      -0.099       0.019
X_train[1]      0.0063      0.015      0.427      0.670      -0.023       0.035
X_train[2]      0.0221      0.019      1.149      0.251      -0.016       0.060
X_train[3]      0.0296      0.010      2.949      0.003       0.010       0.049
X_train[4]     -0.0846      0.022     -3.823      0.000      -0.128      -0.041
X_train[5]      0.0723      0.013      5.500      0.000       0.046       0.098
X_train[6]      0.0003      0.017      0.018      0.986      -0.034       0.034
X_train[7]     -0.1155      0.022     -5.274      0.000      -0.159      -0.072
X_train[8]      0.0644      0.032      2.032      0.043       0.002       0.127
X_train[9]     -0.0871      0.028     -3.097      0.002      -0.142      -0.032
X_train[10]    -0.0707      0.014     -5.013      0.000      -0.098      -0.043
X_train[11]    -0.0058      0.011     -0.531      0.596      -0.027       0.016
X_train[12]    -0.2368      0.016    -14.916      0.000      -0.268      -0.206
==============================================================================
Omnibus:                       29.535   Durbin-Watson:                   2.018
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              110.913
Skew:                           0.106   Prob(JB):                     8.23e-25
Kurtosis:                       5.558   Cond. No.                         11.4
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### 6.2 Polynomial Regression
Extends Linear Regression by adding interaction and squared terms (degree=2).
Captures non-linear relationships between features and house price.
fit_transform on train — transform only on test to prevent data leakage.

In [73]:
#data preprocessing on train data
from sklearn.preprocessing import PolynomialFeatures
polynomial_converter = PolynomialFeatures(degree=2)
X_train_poly= pd.DataFrame(polynomial_converter.fit_transform(X_train))

#Modelling On train Data
from sklearn.linear_model import LinearRegression
model=LinearRegression()
model.fit(X_train_poly,y_train)

#prediction on train data
ypred_train=model.predict(X_train_poly)

#Evaluation on train data
from sklearn.metrics import r2_score
print("Train R2:",r2_score(y_train,ypred_train))

#cross Validation on train data
from sklearn.model_selection import cross_val_score
print("cross Validation score:",cross_val_score(model,X_train_poly,y_train,cv=5).mean())

#prediction on test data
X_test_poly=pd.DataFrame(polynomial_converter.transform(X_test))
ypred_test = model.predict(X_test_poly)

from sklearn.metrics import r2_score
print("test R2:",r2_score(y_test,ypred_test))

Train R2: 0.9301670070274934
cross Validation score: 0.8335032565213647
test R2: 0.7724018266426678


### 6.3 Lasso Regression (L1 Regularization)
Adds L1 penalty to Linear Regression — shrinks less important feature coefficients to zero.
Useful for automatic feature selection.
GridSearchCV used to find best alpha (regularization strength).

In [77]:
from sklearn.model_selection import GridSearchCV

#model
from sklearn.linear_model import Lasso
estimator = Lasso()

#parameters and values
param_grid={"alpha":list(range(1,100))}

#identifying the best value of the parameter within given values for the given data
model_hp=GridSearchCV(estimator,param_grid,cv=5,scoring="r2")

model_hp.fit(X_train,y_train)

model_hp.best_params_

{'alpha': 1}

In [79]:
#modelling
from sklearn.linear_model import Lasso
lasso_best=Lasso(alpha=1)
lasso_best.fit(X_train,y_train)

print("Intercept:",lasso_best.intercept_)
print("coefficients:",lasso_best.coef_)

#prediction and Evaluation on Train data
ypred_train=lasso_best.predict(X_train)

from sklearn.metrics import r2_score
print("train r2",r2_score(y_train,ypred_train))

from sklearn.model_selection import cross_val_score
print("CV Score:",cross_val_score(lasso_best,X_train,y_train,cv=5).mean())

# prediction and evaluation on test data
ypred_test=lasso_best.predict(X_test)
print("test r2:",r2_score(y_test,ypred_test))

Intercept: 3.0891510213245055
coefficients: [-0.  0. -0.  0. -0.  0. -0.  0. -0. -0. -0. -0. -0.]
train r2 0.0
CV Score: -0.0037881129828801095
test r2: -0.0015301272731063076


### 6.4 Ridge Regression (L2 Regularization)
Adds L2 penalty — shrinks all coefficients but never to exactly zero.
Better than Lasso when all features are somewhat relevant.
GridSearchCV used to find best alpha.

In [85]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

estimator = Ridge()

#parameters and values
param_grid={"alpha":list(range(1,100))}

#identifying the best value of the parameter within given values for the given data
model_hp=GridSearchCV(estimator,param_grid,cv=5,scoring="r2")

model_hp.fit(X_train,y_train)

model_hp.best_params_

{'alpha': 1}

In [87]:
#Modelling 
ridge_best = Ridge(alpha=1)
ridge_best.fit(X_train,y_train)

print("Intercept:",ridge_best.intercept_)
print("coefficients:",ridge_best.coef_)

Intercept: 3.0891510213245055
coefficients: [-3.88291387e-02  6.06348809e-03  2.10743478e-02  2.97864015e-02
 -8.33551873e-02  7.29420175e-02  4.83555158e-05 -1.13859810e-01
  6.10086877e-02 -8.43984551e-02 -7.02431578e-02 -5.97929902e-03
 -2.35599460e-01]


In [93]:
# Evaluate the best Ridge model on train, cross-validation and test sets
ypred_train = ridge_best.predict(X_train)
print("Train r2:",r2_score(y_train,ypred_train))
print("CV Score:",cross_val_score(ridge_best,X_train,y_train,cv=5).mean())
ypred_test=ridge_best.predict(X_test)
print("Test r2:",r2_score(y_test,ypred_test))

Train r2: 0.7715167201315092
CV Score: 0.7388611961123951
Test r2: 0.6548962861887683


### 6.5 ElasticNet Regression (L1 + L2 Combined)
Combines both Lasso (L1) and Ridge (L2) penalties.
l1_ratio controls the mix: 0=pure Ridge, 1=pure Lasso.
First trained with default parameters, then tuned with GridSearchCV.

In [95]:
#Modelling
from sklearn.linear_model import ElasticNet
enr_base=ElasticNet()
enr_base.fit(X_train,y_train)

#predictions
train_predictions=enr_base.predict(X_train)
test_predictions=enr_base.predict(X_test)

#evaluation
print("Train r2:",enr_base.score(X_train,y_train))
print("Test r2:",enr_base.score(X_test,y_test)) 
from sklearn.model_selection import cross_val_score
print("CV Score:",cross_val_score(enr_base,X,y,cv=5).mean())

Train r2: 0.0
Test r2: 0.0
CV Score: 0.36743461937639543


In [97]:
from sklearn.model_selection import GridSearchCV

# model
estimator=ElasticNet()

#parameters and values
param_grid={"alpha":[0.1,0.2,1,2,3,5,10],"l1_ratio":[0.1,0.5,0.75,0.9,0.95,1]}

#identifying the best value of the parameter within given values for the given data
model_hp=GridSearchCV(estimator,param_grid,cv=5,scoring="neg_mean_squared_error")
model_hp.fit(X_train,y_train)
model_hp.best_params_

{'alpha': 0.1, 'l1_ratio': 0.1}

In [101]:
##modelling
enr_best=ElasticNet(alpha=0.1,l1_ratio=0.1)
enr_best.fit(X_train,y_train)

print("intercept:",enr_best.intercept_)
print("coefficients:",enr_best.coef_)

#predictions
train_predictions=enr_best.predict(X_train)
test_predictions=enr_best.predict(X_test)

#evaluation
print("Train R2:",enr_best.score(X_train,y_train))
print("test r2:",enr_best.score(X_test,y_test))
print("cross Validation Score:",cross_val_score(enr_best,X,y,cv=5).mean())

intercept: 3.0891510213245055
coefficients: [-0.01402195  0.         -0.          0.0252588  -0.02613671  0.08352015
 -0.         -0.03156647 -0.         -0.03756138 -0.05437149 -0.00339449
 -0.19822611]
Train R2: 0.7500167435497852
test r2: 0.6731768106521341
cross Validation Score: 0.5308760327251715


### 6.6 Decision Tree Regressor
Non-linear tree based model that splits data on feature thresholds.
No hyperparameter tuning applied — baseline performance checked first.
Train R2=1.0 is expected (overfitting) — CV score reveals true performance.

In [103]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

# model
dt_model = DecisionTreeRegressor(random_state=42)

# train
dt_model.fit(X_train, y_train)

# prediction on train
y_pred_train = dt_model.predict(X_train)
print("Train R2:", r2_score(y_train, y_pred_train))

# cross validation
print("CV Score:", cross_val_score(dt_model, X_train, y_train, cv=5, scoring="r2").mean())

# prediction on test
y_pred_test = dt_model.predict(X_test)
print("Test R2:", r2_score(y_test, y_pred_test))

Train R2: 1.0
CV Score: 0.7463505258540301
Test R2: 0.3903474993678002


### 6.7 Random Forest Regressor
Ensemble of decision trees using bagging — reduces overfitting of single trees.
Averages predictions from multiple trees for a more stable result.

In [105]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

# model
rf_model = RandomForestRegressor(random_state=42)

# train
rf_model.fit(X_train, y_train)

# prediction on train
y_pred_train = rf_model.predict(X_train)
print("Train R2:", r2_score(y_train, y_pred_train))

# cross validation
print("CV Score:", cross_val_score(rf_model, X_train, y_train, cv=5, scoring="r2").mean())

# prediction on test
y_pred_test = rf_model.predict(X_test)
print("Test R2:", r2_score(y_test, y_pred_test))

Train R2: 0.9812555703617589
CV Score: 0.8536112645299975
Test R2: 0.7256936464331356


### 6.8 KNN Regressor
Predicts house price by averaging the values of the k nearest neighbors.
n_neighbors=5 used as default — can be tuned with GridSearchCV.

In [107]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

# model
knn_model = KNeighborsRegressor(n_neighbors=5)

# train
knn_model.fit(X_train, y_train)

# prediction on train
y_pred_train = knn_model.predict(X_train)
print("Train R2:", r2_score(y_train, y_pred_train))

# cross validation
print("CV Score:", cross_val_score(knn_model, X_train, y_train, cv=5, scoring="r2").mean())

# prediction on test
y_pred_test = knn_model.predict(X_test)
print("Test R2:", r2_score(y_test, y_pred_test))

Train R2: 0.8504146487739183
CV Score: 0.758247921066981
Test R2: 0.748982227774761


## 7. Prediction on New House
Applying the full preprocessing pipeline to a new unseen house sample.
Steps applied in the same order as training:
1. Log transform skewed features (CRIM, ZN, DIS, B)
2. Scale using the same StandardScaler fitted on training data
3. Apply Polynomial Features (degree=2)
4. Predict using the trained Linear Regression model
5. Reverse log transform using np.expm1() to get actual price in $1000s

In [111]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# load data
df = pd.read_csv("Housing.csv")

# remove duplicates
df = df.drop_duplicates()

# fill missing values
df = df.fillna(df.median(numeric_only=True))

# save original max of B before transformation
B_max = df['B'].max()

# skewness handling
df['CRIM'] = np.log1p(df['CRIM'])
df['ZN'] = np.log1p(df['ZN'])
df['DIS'] = np.log1p(df['DIS'])
df['B'] = np.log1p(B_max - df['B'])
df['MEDV'] = np.log1p(df['MEDV'])

# split features and target
X = df.drop('MEDV', axis=1)
y = df['MEDV']

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# train model
model = LinearRegression()
model.fit(X_train_poly, y_train)

# evaluation
y_pred_train = model.predict(X_train_poly)
y_pred_test = model.predict(X_test_poly)

print("Train R2:", r2_score(y_train, y_pred_train))
print("Test R2:", r2_score(y_test, y_pred_test))

# -------- prediction on new data --------
# feature order:
# CRIM, ZN, INDUS, CHAS, NOX, RM, AGE, DIS, RAD, TAX, PTRATIO, B, LSTAT

new_data = [[0.1, 18, 2.3, 0, 0.45, 6.5, 65, 4.5, 1, 300, 15.3, 390, 5]]

# apply same preprocessing to new input
new_data[0][0] = np.log1p(new_data[0][0])      # CRIM
new_data[0][1] = np.log1p(new_data[0][1])      # ZN
new_data[0][7] = np.log1p(new_data[0][7])      # DIS
new_data[0][11] = np.log1p(B_max - new_data[0][11])  # B

new_data = np.array(new_data)

# scale and transform
new_data_scaled = scaler.transform(new_data)
new_data_poly = poly.transform(new_data_scaled)

# predict
prediction_log = model.predict(new_data_poly)
prediction_actual = np.expm1(prediction_log)

print("Predicted MEDV in log scale:", prediction_log)
print("Predicted MEDV in original scale:", prediction_actual)

Train R2: 0.9324806183552544
Test R2: 0.8001477231952909
Predicted MEDV in log scale: [3.23687751]
Predicted MEDV in original scale: [24.45411728]


C:\Users\zainh\anaconda3\envs\ml_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
